# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

**Note:** Use `dataset.metadata.record_sets` to access record sets. For each `RecordSet`, review its `@id`, `name`, and associated `field`s and `column`s (all by `@id`).

In [ ]:
# List all record sets and their fields
record_sets = list(dataset.metadata.record_sets)
print(f"Number of Record Sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Fields:")
    for f in rs.fields:
        print(f"    - Field name: {f.name}, @id: {f.id}, dataType: {f.data_type}")
    print(f"  Columns:")
    for c in rs.columns:
        print(f"    - Column: {c.name}, @id: {c.id}")
    print("-")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set by @id
# (You may update the selected record set @ids based on the above overview)
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Extract rows for each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Example: print column names and show rows for the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"Record set: {first_rs}")
    print("Columns:", dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section demonstrates filtering a numeric field, normalization, and grouping. Make sure to update `numeric_field_id` and, if grouping, `group_field_id` to match field `@id`s found in the overview.

In [ ]:
# Specify the Record Set and field @ids you want to analyze
# Example selection from the first detected record set
record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[record_set_id]

# Use a numeric field from the first record set (update as needed after printout from previous cell)
numeric_field_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
print("Numeric field candidates:", numeric_field_candidates)
numeric_field = numeric_field_candidates[0] if numeric_field_candidates else df.columns[0]

threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Optionally, group by a categorical field
group_field_candidates = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]
if group_field_candidates:
    group_field = group_field_candidates[0]
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field} (mean of numeric fields):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example histogram of the selected numeric field, and a boxplot by group if grouping is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# If there is a group field, plot boxplot
if group_field_candidates:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset's record sets, fields, and IDs were successfully explored via Croissant schema using `mlcroissant`.
- Sample numeric fields were filtered, normalized, and visualized.
- This workflow can be adapted to any Croissant-compliant dataset using field/record set `@id`s.